# Week 8 — Day 3: ScannerAgent + MessagingAgent

Two more agents in the system:
1. **ScannerAgent** — pulls deals from RSS feeds and uses GPT-5-mini to pick the 5 most-detailed promising ones with structured output
2. **MessagingAgent** — sends push notifications via Pushover when a great deal is identified

## Setup

In [ ]:
import os
import logging
import requests
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection

load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

## Fetch raw deals from RSS feeds

In [ ]:
deals = ScrapedDeal.fetch(show_progress=True)

In [ ]:
len(deals)

In [ ]:
deals[10].describe()

## Use GPT-5-mini with structured output to pick the 5 best deals

`response_format=DealSelection` (a Pydantic model) forces the API to return valid JSON in our schema.

In [ ]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price.
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price.

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [ ]:
def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [ ]:
user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_prompt}
]

In [ ]:
response = openai.chat.completions.parse(
    model=MODEL,
    messages=messages,
    response_format=DealSelection,
    reasoning_effort="minimal"
)
results = response.choices[0].message.parsed
results

In [ ]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()

## Wrap it in the ScannerAgent class

In [ ]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
from agents.scanner_agent import ScannerAgent

In [ ]:
agent = ScannerAgent()
result = agent.scan()

In [ ]:
result

## Pushover — push notifications to your phone

1. Sign up at https://pushover.net/ (free)
2. From the home screen, click **Create an Application/API Token**, name it anything, click Create
3. Add to `.env`:
   ```
   PUSHOVER_USER=u...   # the user key on the top-right of your Pushover home screen
   PUSHOVER_TOKEN=a...  # the token from the application page
   ```
4. Save `.env` and rerun `load_dotenv(override=True)`
5. Install the Pushover app on your phone ("Add Phone, Tablet or Desktop")

In [ ]:
load_dotenv(override=True)

In [ ]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

print(f"Pushover user found: {bool(pushover_user)}")
print(f"Pushover token found: {bool(pushover_token)}")

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("Test push from the deal-hunting agent")

## MessagingAgent — wraps Pushover behind a clean interface

In [ ]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("Test push via MessagingAgent")

In [ ]:
agent.notify(
    "A special deal on Samsung 60-inch LED TV going at a great bargain",
    300,
    1000,
    "https://www.samsung.com"
)